# Bagging classifier
BaggingClassifier on a synthetic dataset of 10,000 rows: bagging, bagged SVMs, pasting, random subspaces,
random patches, the out-of-bag score and a grid search. The decision-surface demo is `app.py`
(`python app.py`, then open http://127.0.0.1:8050).

In [1]:
import time

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 1. Data and a single tree

In [2]:
# 10,000 rows, 10 columns, of which 3 carry information
X, y = make_classification(n_samples=10000, n_features=10, n_informative=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("single decision tree:", accuracy_score(y_test, dt.predict(X_test)))

(8000, 10) (2000, 10)
single decision tree: 0.9265


## 2. Bagging: 500 trees, each on 25% of the rows, drawn with replacement
In scikit-learn 1.9 the base model is passed as `estimator` (older code used `base_estimator`).

In [3]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,      # 25% of 8,000 rows = 2,000 rows per tree
    bootstrap=True,        # with replacement
    random_state=42,
    n_jobs=-1,             # use every CPU core
)
bag.fit(X_train, y_train)
print("bagging:", accuracy_score(y_test, bag.predict(X_test)))

bagging: 0.945


In [4]:
# Which rows and columns did the first tree get?
print("rows for tree 1:", bag.estimators_samples_[0].shape, "first few:", bag.estimators_samples_[0][:8])
print("columns for tree 1:", bag.estimators_features_[0])

rows for tree 1: (2000,) first few: [2523 3113 7114 6476 2690 2953 3303 6748]
columns for tree 1: [0 1 2 3 4 5 6 7 8 9]


In [5]:
# The same with max_samples=0.5: 4,000 rows per tree
bag_half = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.5, random_state=42, n_jobs=-1)
bag_half.fit(X_train, y_train)
print("bagging, 50% rows:", accuracy_score(y_test, bag_half.predict(X_test)))
print("rows for tree 1:", bag_half.estimators_samples_[0].shape)

bagging, 50% rows: 0.95


rows for tree 1: (4000,)


## 3. Bagging SVMs

In [6]:
bag_svm = BaggingClassifier(estimator=SVC(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            random_state=42, n_jobs=-1)
bag_svm.fit(X_train, y_train)
print("bagging with SVM:", accuracy_score(y_test, bag_svm.predict(X_test)))

bagging with SVM: 0.9125


## 4. Pasting: rows without replacement (bootstrap=False); verbose=1 prints progress

In [7]:
paste = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=False,
                          random_state=42, verbose=1, n_jobs=-1)
paste.fit(X_train, y_train)
print("pasting:", accuracy_score(y_test, paste.predict(X_test)))

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    2.6s remaining:   13.1s


[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    4.8s finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    0.2s remaining:    0.8s


pasting: 0.946


[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    0.7s finished


## 5. Random subspaces: all rows, half the columns

In [8]:
subspace = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=1.0, bootstrap=False,
                             max_features=0.5, bootstrap_features=True, random_state=42, n_jobs=-1)
subspace.fit(X_train, y_train)
print("random subspaces:", accuracy_score(y_test, subspace.predict(X_test)))
print("rows for tree 1:", subspace.estimators_samples_[0].shape, "| columns for tree 1:", subspace.estimators_features_[0])

random subspaces: 0.9415


rows for tree 1: (8000,) | columns for tree 1: [9 2 9 7 7]


## 6. Random patches: 25% of the rows and half the columns

In [9]:
patches = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            max_features=0.5, bootstrap_features=True, random_state=42, n_jobs=-1)
patches.fit(X_train, y_train)
print("random patches:", accuracy_score(y_test, patches.predict(X_test)))

random patches: 0.938


## 7. Out-of-bag score
Each tree is scored on the training rows its sample missed; oob_score_ averages these into one accuracy.

In [10]:
bag_oob = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            oob_score=True, random_state=42, n_jobs=-1)
bag_oob.fit(X_train, y_train)
print("out-of-bag score:", round(bag_oob.oob_score_, 4))
print("test accuracy:   ", accuracy_score(y_test, bag_oob.predict(X_test)))

out-of-bag score: 0.9429


test accuracy:    0.945


## 8. Grid search over the bagging hyperparameters

In [11]:
parameters = {
    "n_estimators": [50, 100, 500],
    "max_samples": [0.1, 0.4, 0.7, 1.0],
    "bootstrap": [True, False],
    "max_features": [0.1, 0.4, 0.7, 1.0],
}
start = time.time()
search = GridSearchCV(BaggingClassifier(random_state=42), parameters, cv=5, n_jobs=-1)
search.fit(X_train, y_train)
print(f"{len(search.cv_results_['params'])} combinations in {time.time() - start:.0f} s")
print("best parameters:", search.best_params_)
print("best cross-validation accuracy:", round(search.best_score_, 4))
print("test accuracy:", search.score(X_test, y_test))

96 combinations in 1729 s
best parameters: {'bootstrap': False, 'max_features': 0.7, 'max_samples': 0.7, 'n_estimators': 500}
best cross-validation accuracy: 0.9546


test accuracy: 0.952


## 9. The decision-surface demo (same code as app.py)

In [12]:
from app import figure, fit

(single, a1), (bagged, a2) = fit("decision tree", n_estimators=100, max_samples=50)
figure([(f"single tree: {a1:.3f}", single), (f"bagging: {a2:.3f}", bagged)])

In [13]:
# Bagging vs pasting: change only `bootstrap`, measure tree correlation, ensemble variance and bias (sine data)
import numpy as np
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
rng = np.random.default_rng(0)
x_line = np.linspace(0, 6, 400)[:, None]
truth = np.sin(x_line.ravel())
sets = []
for _ in range(100):
    x = rng.uniform(0, 6, 60)
    sets.append((x[:, None], np.sin(x) + rng.normal(0, 0.4, 60)))
for bootstrap in (True, False):
    preds, corrs = [], []
    for Xs, ys in sets:
        m = BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, max_samples=0.5,
                             bootstrap=bootstrap, random_state=0).fit(Xs, ys)
        preds.append(m.predict(x_line))
        trees = np.array([e.predict(x_line) for e in m.estimators_])
        corrs.append(np.corrcoef(trees)[np.triu_indices(100, 1)].mean())
    P = np.array(preds)
    print(f"bootstrap={bootstrap}: tree correlation {np.mean(corrs):.3f}, "
          f"variance {P.var(0).mean():.4f}, bias² {((P.mean(0) - truth) ** 2).mean():.4f}")

bootstrap=True: tree correlation 0.821, variance 0.0413, bias² 0.0019


bootstrap=False: tree correlation 0.842, variance 0.0548, bias² 0.0013
